# 🏨 Hotels in Berlin — Data Transformation & Enrichment

## Objective
This notebook builds a clean, standardized **Hotels in Berlin** dataset starting from raw OpenStreetMap (OSM) data and enriching it with selected external sources.

The final output is a geospatial dataset (`gdf_final`) that:
- Represents hotels and hotel-like accommodations in Berlin
- Uses a unified point-based geometry model (WGS84(EPSG:4326))
- Follows the shared POI schema used across layers
- Tracks provenance and source attribution explicitly

## Scope & locked decisions

This notebook focuses on **data transformation and enrichment** only.
Exploration, modelling decisions, and data quality trade-offs are documented explicitly.

### Locked decisions
- **Primary source:** OpenStreetMap (OSM)
- **Geometry:** Point geometry only  
  (OSM ways/polygons are converted to centroids)
- **Identifier strategy:**
  - `id` is the original OSM element id
  - Provenance is tracked via `data_source` and `source_ids`
- **Amenities & accessibility:**  
  Derived exclusively from raw OSM tags, then assigned to the working dataset
- **Star ratings:**  
  Enriched from Wikidata property **P10290** (hotel rating → stars)  
  → Only used to fill missing values (no overwrite)
- **Address completion:**  
  Missing address components are filled via Nominatim reverse geocoding  
  → Existing OSM-derived values are never overwritten
- **Deduplication:**  
  Known node/way duplicate pairs are resolved deterministically

### Explicitly out of scope
- Website enrichment via Wikidata  
  (coverage too small to justify complexity)
- Commercial hotel platforms (licensing constraints)
- Any attempt to infer or predict missing attributes

## Step 1 — Data sources & modelling decisions

This step documents the data sources used to build the **Hotels in Berlin** layer,
the role each source plays in the final dataset, and the modelling decisions derived
from their structure and coverage.

Only sources that directly contribute to the final dataset are described in detail.
Sources that were evaluated but not used are listed separately with justification.

### Target entity & schema overview

- **Entity:** Hotels and hotel-like accommodations in Berlin
- **Spatial representation:** Point geometry (WGS84 / EPSG:4326)
- **Primary identifier:**
  - `id` — original OpenStreetMap element id (node or way)
- **Provenance tracking:**
  - `data_source` — primary origin of the record
  - `source_ids` — contributing identifiers across sources
- **Core attributes:**
  - name
  - hotel_type
  - star_rating
  - amenities
  - accessibility
- **Address attributes:**
  - street
  - house_number
  - postal_code
  - city
- **Berlin administrative context:**
  - district
  - district_id
  - neighborhood
  - neighborhood_id

The final dataset follows the shared POI schema used across layers,
with hotels-specific attributes added where relevant.

### Data sources used

#### Source 1 — OpenStreetMap (OSM)

- **Origin:** OpenStreetMap
- **Access method:** OSMnx / Overpass API
- **Update frequency:** Continuous (community-maintained)
- **Data type:** Dynamic
- **Role in dataset:**
  - Primary source for hotel locations and attributes
- **Fields used:**
  - name
  - tourism / accommodation type
  - address tags
  - contact tags
  - website
  - raw tags for amenities and accessibility
  - geometry (node or way)
- **Notes:**
  - The original OSM element id is preserved as the dataset `id`
  - Raw tag columns are used only during transformation and are not exposed in the final output

#### Source 2 — LOR Ortsteile (Berlin administrative areas)

- **Origin:** Berlin LOR Ortsteile GeoJSON
- **Update frequency:** Infrequent
- **Data type:** Mostly static
- **Role in dataset:**
  - Spatial enrichment of hotels with Berlin administrative context
- **Fields used:**
  - district name and identifier
  - neighborhood name and identifier
- **Method:**
  - Spatial join between hotel point geometries and LOR polygons

#### Source 3 — Wikidata (hotel star ratings)

- **Origin:** Wikidata
- **Access method:** SPARQL query (results persisted as a CSV snapshot)
- **Property used:** P10290 (hotel rating → stars)
- **Update frequency:** Community-maintained
- **Data type:** Dynamic source, static snapshot used in pipeline
- **Role in dataset:**
  - Enrichment of missing `star_rating` values only
- **Rules applied:**
  - Existing OSM star ratings are never overwritten
  - Only numeric star values in the 1–5 range are accepted

#### Source 4 — Nominatim (reverse geocoding)

- **Origin:** Nominatim / OpenStreetMap
- **Access method:** Reverse geocoding API
- **Update frequency:** Continuous
- **Data type:** Dynamic
- **Role in dataset:**
  - Fill missing address components
- **Rules applied:**
  - Only missing address fields are populated
  - Existing OSM-derived address values are never overwritten
  - Requests are rate-limited and cached where possible

### Sources considered but not used

- **Wikidata (website ,amenities enrichment):**
  - Only a small number of usable candidates were identified
  - Coverage was insufficient to justify additional complexity
- **Commercial hotel platforms (e.g. Booking, Expedia, Google):**
  - Licensing and redistribution constraints
- **Other tourism datasets:**
  - Overlapping coverage with OSM or incompatible schemas

### Modelling & transformation principles

- OpenStreetMap is treated as the single authoritative base layer
- External sources are used strictly for enrichment
- Geometry is standardized to a point representation
- Administrative context is added via spatial joins
- Amenities and accessibility are derived from structured OSM tags
- Duplicate node/way representations are resolved deterministically
- All transformation decisions are evidence-based and traceable

## Step 2 — Transformation & enrichment pipeline

This section implements the transformation pipeline described in Step 1.
We first extract the raw OSM hotels dataset (`gdf_osm_raw`) and keep it unchanged.
All transformations are applied to derived working datasets.

### Setup & configuration

This notebook uses:
- OpenStreetMap (OSM) as the primary data source
- LOR Ortsteile polygons for Berlin administrative enrichment
- Wikidata (P10290) for star rating enrichment (cached as CSV)
- Nominatim reverse geocoding to fill missing address components (no overwrite)

Key configuration:
- CRS is standardized to WGS84 / EPSG:4326
- External calls (Wikidata/Nominatim) are toggleable for reproducibility

In [1]:
import re
import time
from datetime import datetime

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import osmnx as ox
import requests

In [ ]:
# -----------------------------
# Core configuration
# -----------------------------
PLACE_NAME = "Berlin, Germany"
TARGET_CRS = "EPSG:4326"

# OSM tags for hotels / accommodation-like POIs
OSM_TAGS = {
    "tourism": [
        "hotel",
        "hostel",
        "guest_house",
        "aparthotel",
        "motel"
    ]
}

# -----------------------------
# Local file paths (user-provided)
# -----------------------------
LOR_PATH = "../mapping/lor_ortsteile.geojson"
WIKIDATA_STARS_PATH = "../hotels/sources/wikidata_stars_candidates.csv"

# -----------------------------
# Nominatim config (polite usage)
# -----------------------------
NOMINATIM_URL = "https://nominatim.openstreetmap.org/reverse"
NOMINATIM_SLEEP_SEC = 1.0
NOMINATIM_USER_AGENT = "webeet-hotels-berlin-transformation (contact: ..)"


### Extract raw hotels from OSM

We extract the raw dataset from OpenStreetMap using OSMnx and keep it unchanged as `gdf_osm_raw`.
This raw dataset is used for:
- Exploratory analysis (EDA)
- Tag-derived attributes (amenities & accessibility)
- Building the working transformation dataset

In [3]:
# Raw extraction (do not modify this object downstream)
gdf_osm_raw = ox.features_from_place(PLACE_NAME, tags=OSM_TAGS)

print("✅ Raw OSM hotels extracted.")
print("Shape:", gdf_osm_raw.shape)
print("CRS:", gdf_osm_raw.crs)
gdf_osm_raw.head()

✅ Raw OSM hotels extracted.
Shape: (881, 217)
CRS: epsg:4326


geometry                          name  \
element id                                                                  
node    29997721   POINT (13.27667 52.5015)                  Rasthof AVUS   
        58467591   POINT (13.4078 52.51146)         Park Plaza Wallstreet   
        60105926  POINT (13.40879 52.51225)  Living Hotel Großer Kurfürst   
        60105927  POINT (13.41034 52.51254)         art’otel Berlin Mitte   
        60105928  POINT (13.40825 52.51208)     Living Hotel Berlin-Mitte   

                 tourism addr:city addr:country addr:housenumber  \
element id                                                         
node    29997721   motel       NaN          NaN              NaN   
        58467591   hotel    Berlin           DE               23   
        60105926   hotel    Berlin           DE            11-12   
        60105927   hotel    Berlin           DE            70-73   
        60105928   hotel    Berlin           DE               13   

                 addr:postcode     addr:street addr:suburb       brand  ...  \
element id                                                              ...   
node    29997721           NaN             NaN         NaN         NaN  ...   
        58467591         10179      Wallstraße       Mitte  Park Plaza  ...   
        60105926         10179  Neue Roßstraße       Mitte         NaN  ...   
        60105927         10179      Wallstraße       Mitte    Art'otel  ...   
        60105928         10179  Neue Roßstraße       Mitte         NaN  ...   

                 type loc_name architect:wikipedia branch:en check_in dog:fee  \
element id                                                                      
node    29997721  NaN      NaN                 NaN       NaN      NaN     NaN   
        58467591  NaN      NaN                 NaN       NaN      NaN     NaN   
        60105926  NaN      NaN                 NaN       NaN      NaN     NaN   
        60105927  NaN      NaN                 NaN       NaN      NaN     NaN   
        60105928  NaN      NaN                 NaN       NaN      NaN     NaN   

                 note:payment:cash payment:others source:access:password  \
element id                                                                 
node    29997721               NaN            NaN                    NaN   
        58467591               NaN            NaN                    NaN   
        60105926               NaN            NaN                    NaN   
        60105927               NaN            NaN                    NaN   
        60105928               NaN            NaN                    NaN   

                 source:internet_access:password  
element id                                        
node    29997721                             NaN  
        58467591                             NaN  
        60105926                             NaN  
        60105927                             NaN  
        60105928                             NaN  

[5 rows x 217 columns]

## Exploratory Data Analysis (EDA)

This exploratory analysis provides a high-level understanding of the raw OpenStreetMap hotels data for Berlin.

The goal is to assess:
- dataset structure and size
- geometry types (nodes vs ways)
- attribute availability and missingness
- coverage of star ratings and Wikidata ids

In [4]:
# 3) Inspection only: index, element types, columns, missingness

print("=== Shape ===")
print("Rows:", len(gdf_osm_raw))
print("Columns:", len(gdf_osm_raw.columns))

print("\n=== CRS ===")
print(gdf_osm_raw.crs)

print("\n=== Index info ===")
print("Index type:", type(gdf_osm_raw.index))
print("Index names:", gdf_osm_raw.index.names)

# Element type counts (works if MultiIndex contains 'element')
if isinstance(gdf_osm_raw.index, pd.MultiIndex) and "element" in gdf_osm_raw.index.names:
    print("\n=== Element counts (from index) ===")
    print(gdf_osm_raw.index.get_level_values("element").value_counts())
else:
    print("\n(No 'element' level found in index.)")

print("\n=== Full column list ===")
cols = list(gdf_osm_raw.columns)
print(cols)

print("\n=== Top 20 columns by non-null count ===")
non_null_counts = gdf_osm_raw.notna().sum().sort_values(ascending=False)
print(non_null_counts.head(20))

print("\n=== Bottom 20 columns by non-null count ===")
print(non_null_counts.tail(20))

=== Shape ===
Rows: 881
Columns: 217

=== CRS ===
epsg:4326

=== Index info ===
Index type: <class 'pandas.core.indexes.multi.MultiIndex'>
Index names: ['element', 'id']

=== Element counts (from index) ===
element
node        627
way         242
relation     12
Name: count, dtype: int64

=== Full column list ===
['geometry', 'name', 'tourism', 'addr:city', 'addr:country', 'addr:housenumber', 'addr:postcode', 'addr:street', 'addr:suburb', 'brand', 'brand:wikidata', 'brand:wikipedia', 'stars', 'wheelchair', 'wikidata', 'contact:email', 'contact:phone', 'contact:website', 'operator', 'toilets:wheelchair', 'check_date', 'old_name', 'website', 'wikimedia_commons', 'air_conditioning', 'internet_access', 'rooms', 'email', 'fax', 'image', 'bar', 'contact:facebook', 'internet_access:fee', 'opening_hours', 'payment:cash', 'payment:credit_cards', 'payment:maestro', 'payment:mastercard', 'payment:visa', 'reservation', 'smoking', 'wikipedia', 'phone', 'branch', 'payment:debit_cards', 'opening_hour

### Geometry inspection

OSM hotels can be represented as:
- nodes (Point)
- ways (Polygon / MultiPolygon)

We inspect the geometry type distribution to justify our later standardization to point geometries.

In [5]:
geom_counts = gdf_osm_raw.geometry.geom_type.value_counts(dropna=False)
geom_pct = (geom_counts / geom_counts.sum() * 100).round(2)

geom_summary = pd.DataFrame({"count": geom_counts, "pct": geom_pct})
geom_summary

,count,pct
Point,627,71.17
Polygon,252,28.60
MultiPolygon,2,0.23


### Attribute availability (key fields)

We inspect missingness for a focused set of key fields to understand:
- baseline completeness from OSM
- where enrichment is required (Wikidata stars, Nominatim address completion)

In [6]:
key_cols = [
    "name",
    "tourism",
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
    "contact:website",
    "website",
    "contact:phone",
    "phone",
    "contact:email",
    "email",
    "stars",
    "wikidata",
]

present_key_cols = [c for c in key_cols if c in gdf_osm_raw.columns]

missing = gdf_osm_raw[present_key_cols].isna().mean().sort_values(ascending=False).mul(100).round(2)
missing.to_frame("missing_%")

,missing_%
contact:email,87.85
stars,85.24
email,84.34
wikidata,78.21
contact:phone,73.89
phone,71.51
contact:website,70.72
website,56.98
addr:city,21.57
addr:postcode,20.09


### Address completeness

Address information in OpenStreetMap is often partially populated.
We inspect individual address components to determine:
- which fields are most frequently missing
- whether a conservative reverse geocoding strategy is justified

In [7]:
address_cols = [
    "addr:street",
    "addr:housenumber",
    "addr:postcode",
    "addr:city",
]

present_address_cols = [c for c in address_cols if c in gdf_osm_raw.columns]

addr_missing = (
    gdf_osm_raw[present_address_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

addr_missing.to_frame("missing_%")

,missing_%
addr:city,21.57
addr:postcode,20.09
addr:housenumber,18.73
addr:street,17.93


### Hotel classification (OSM `tourism` tag)

In this dataset, hotels and hotel-like accommodations are explicitly selected
using a filtered `tourism` tag during extraction.

The extraction includes the following `tourism` values:
- hotel
- hostel
- guest_house
- aparthotel
- motel

We inspect the distribution of these values to:
- validate the extraction filter
- understand the composition of accommodation types
- inform normalization into a unified `hotel_type`

In [8]:
# Distribution of hotel-related tourism values
(
    gdf_osm_raw["tourism"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
tourism,
hotel,647
hostel,127
guest_house,102
motel,5


### Star rating coverage

Star ratings are inconsistently provided in OpenStreetMap.
We inspect:
- how many hotels provide an OSM `stars` value
- how many hotels reference a Wikidata entity (QID),
  which enables secondary enrichment

In [9]:
coverage = {}

if "stars" in gdf_osm_raw.columns:
    coverage["osm_stars_present_%"] = round(
        gdf_osm_raw["stars"].notna().mean() * 100, 2
    )

if "wikidata" in gdf_osm_raw.columns:
    coverage["wikidata_qid_present_%"] = round(
        gdf_osm_raw["wikidata"].notna().mean() * 100, 2
    )

pd.DataFrame.from_dict(coverage, orient="index", columns=["coverage_%"])

,coverage_%
osm_stars_present_%,14.76
wikidata_qid_present_%,21.79


### OSM tag signals for amenities and accessibility

Beyond core attributes, OpenStreetMap encodes additional information
using structured tags related to amenities and accessibility
(e.g. wheelchair access, internet availability).

We perform a lightweight inspection of selected tag keys to verify:
- whether such tags are present in the dataset
- whether they appear with sufficient frequency to justify downstream derivation

In [10]:
# Inspect presence of selected amenity & accessibility-related OSM tags
tag_keys = [
    "wheelchair",
    "internet_access",
    "internet_access:fee",
    "toilets:wheelchair",
    "parking",
    "parking:disabled",
    "rooms",
]

tag_coverage = {
    tag: int(gdf_osm_raw[tag].notna().sum())
    for tag in tag_keys
    if tag in gdf_osm_raw.columns
}

pd.DataFrame.from_dict(
    tag_coverage,
    orient="index",
    columns=["non_null_count"]
).sort_values("non_null_count", ascending=False)

,non_null_count
wheelchair,564
internet_access,289
internet_access:fee,138
toilets:wheelchair,122
rooms,101


### EDA summary & implications

Key observations from the exploratory analysis:

- OpenStreetMap provides strong spatial coverage for hotels in Berlin,
  but attribute completeness varies significantly
- Hotel geometries are represented as both points and polygons,
  requiring standardization to a point-based model
- Address components are often partially missing,
  justifying conservative reverse geocoding
- Star ratings are sparsely populated in OSM,
  while Wikidata identifiers enable limited secondary enrichment
- Selected OSM tags related to amenities and accessibility are present in the raw data,
  supporting a dedicated derivation step later in the pipeline

These findings directly inform the transformation and enrichment steps
implemented in the remainder of this notebook.

### MVP column selection

Before applying any transformations, we select a **minimal viable set of columns**
from the raw OSM dataset.

The goal is to:
- keep only fields relevant to the Hotels entity
- reduce noise from high-cardinality or rarely used tags
- establish a stable working dataset for downstream transformations

The raw dataset (`gdf_osm_raw`) remains unchanged.
All transformations are applied to derived copies.

In [11]:
MVP_COLUMNS = [
    # identifiers / classification
    "tourism",
    "name",
    "stars",
    "wikidata",
    "rooms",

    # address components
    "addr:street",
    "addr:housenumber",
    "addr:postcode",

    # contact / web
    "contact:website",
    "website",
    "contact:phone",
    "phone",
    "contact:email",
    "email",

    # raw tag container (needed later for amenities/accessibility)
    "geometry",
]

# Keep only columns that actually exist (OSM schemas vary slightly)
present_mvp_columns = [c for c in MVP_COLUMNS if c in gdf_osm_raw.columns]

gdf_mvp = gdf_osm_raw[present_mvp_columns].copy()

print("MVP shape:", gdf_mvp.shape)
gdf_mvp.head()

MVP shape: (881, 15)


tourism                          name stars    wikidata  \
element id                                                                 
node    29997721   motel                  Rasthof AVUS   NaN         NaN   
        58467591   hotel         Park Plaza Wallstreet     4  Q111388246   
        60105926   hotel  Living Hotel Großer Kurfürst     4  Q111390188   
        60105927   hotel         art’otel Berlin Mitte   NaN  Q111390189   
        60105928   hotel     Living Hotel Berlin-Mitte     4  Q111388522   

                 rooms     addr:street addr:housenumber addr:postcode  \
element id                                                              
node    29997721   NaN             NaN              NaN           NaN   
        58467591   NaN      Wallstraße               23         10179   
        60105926   NaN  Neue Roßstraße            11-12         10179   
        60105927   NaN      Wallstraße            70-73         10179   
        60105928   NaN  Neue Roßstraße               13         10179   

                                                    contact:website website  \
element id                                                                    
node    29997721                                                NaN     NaN   
        58467591                                                NaN     NaN   
        60105926  https://www.living-hotels.com/hotel-grosser-ku...     NaN   
        60105927  http://www.artotels.com/berlin-hotel-de-d-1017...     NaN   
        60105928   https://www.living-hotels.com/hotel-berlin-mitte     NaN   

                    contact:phone phone                       contact:email  \
element id                                                                    
node    29997721              NaN   NaN                                 NaN   
        58467591              NaN   NaN                                 NaN   
        60105926    +49 30 246000   NaN  grosserkurfuerst@living-hotels.com   
        60105927              NaN   NaN                                 NaN   
        60105928  +49 30 24600900   NaN       berlinmitte@living-hotels.com   

                 email                   geometry  
element id                                         
node    29997721   NaN   POINT (13.27667 52.5015)  
        58467591   NaN   POINT (13.4078 52.51146)  
        60105926   NaN  POINT (13.40879 52.51225)  
        60105927   NaN  POINT (13.41034 52.51254)  
        60105928   NaN  POINT (13.40825 52.51208)

### Geometry standardization (points-only)

OpenStreetMap hotels may be represented as:
- nodes (Point geometries)
- ways (Polygon or MultiPolygon geometries)

To ensure a consistent spatial model across the dataset,
all geometries are standardized to **point representations**.

Rules applied:
- Point geometries are kept as-is
- Polygon and MultiPolygon geometries are converted to centroids
- CRS is enforced as WGS84 / EPSG:4326

This step creates a new working dataset with point-only geometry.

In [12]:
# Ensure CRS consistency
if gdf_mvp.crs is None:
    gdf_mvp = gdf_mvp.set_crs(TARGET_CRS)
else:
    gdf_mvp = gdf_mvp.to_crs(TARGET_CRS)

# Create a point-only geometry column
def to_point(geom):
    if geom.geom_type == "Point":
        return geom
    else:
        return geom.centroid

gdf_mvp_points = gdf_mvp.copy()
gdf_mvp_points["geometry"] = gdf_mvp_points.geometry.apply(to_point)

# Sanity check
gdf_mvp_points.geometry.geom_type.value_counts()

Point    881
Name: count, dtype: int64

After this step, all hotel records are represented by point geometries.

This standardized representation enables:
- spatial joins with administrative boundaries
- consistent latitude / longitude extraction
- uniform downstream processing

### Normalize core fields

In this step we create a clean working dataset from the point-standardized MVP layer.

We:
- rename OSM columns into stable working names
- normalize `hotel_type` from the OSM `tourism` tag
- consolidate contact fields (website / phone / email) using precedence rules
- extract latitude / longitude from point geometry

In [13]:
gdf_working = gdf_mvp_points.copy()

rename_map = {
    "tourism": "hotel_type",
    "name": "name",

    # raw stars, wikidata id, rooms
    "stars": "star_rating",
    "wikidata": "wikidata_id",
    "rooms": "room_count",

    # address components (keep only these for now)
    "addr:street": "street",
    "addr:housenumber": "house_number",
    "addr:postcode": "postal_code",

    # contact candidates (we’ll consolidate next)
    "contact:website": "contact_website",
    "website": "website_raw",
    "contact:phone": "contact_phone",
    "phone": "phone_raw",
    "contact:email": "contact_email",
    "email": "email_raw",
}

gdf_working = gdf_working.rename(columns=rename_map)
gdf_working.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 881 entries, ('node', np.int64(29997721)) to ('way', np.int64(1414141828))
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   hotel_type       881 non-null    object  
 1   name             870 non-null    object  
 2   star_rating      130 non-null    object  
 3   wikidata_id      192 non-null    object  
 4   room_count       101 non-null    object  
 5   street           723 non-null    object  
 6   house_number     716 non-null    object  
 7   postal_code      704 non-null    object  
 8   contact_website  258 non-null    object  
 9   website_raw      379 non-null    object  
 10  contact_phone    230 non-null    object  
 11  phone_raw        251 non-null    object  
 12  contact_email    107 non-null    object  
 13  email_raw        138 non-null    object  
 14  geometry         881 non-null    geometry
dtypes: geometry(1), object(14)
mem

### Initialize identifier and provenance (OSM)

We extract the OSM element type and id from the OSMnx index:
- `osm_type` (e.g. node / way)
- `osm_id` (the numeric identifier)

We verify whether `osm_id` is unique across the dataset.
If it is unique, we use it as the stable record identifier `id`.

Then we initialize provenance fields:
- `data_source = "osm"`
- `source_ids = "osm:<id>"`

In [14]:
# Extract OSM components from index (OSMnx MultiIndex: element_type, osmid)
gdf_working["osm_type"] = gdf_working.index.get_level_values(0).astype("string")
gdf_working["osm_id"] = gdf_working.index.get_level_values(1).astype("string")

print("Rows:", len(gdf_working))
print("Unique osm_id count:", gdf_working["osm_id"].nunique())
print("Unique (osm_type, osm_id) pairs:", gdf_working[["osm_type", "osm_id"]].drop_duplicates().shape[0])

# Use osm_id as stable id (as per our earlier validation decision)
gdf_working["id"] = gdf_working["osm_id"]

# Initialize provenance columns
gdf_working["data_source"] = "osm"
gdf_working["source_ids"] = "osm:" + gdf_working["id"]

Rows: 881
Unique osm_id count: 881
Unique (osm_type, osm_id) pairs: 881


In [15]:
gdf_working["id"].duplicated().sum()

np.int64(0)

### Normalize hotel_type

`hotel_type` comes from the filtered OSM `tourism` tag.
We apply minimal normalization to ensure consistent formatting.
Fill the few missing  `names` with  'unknown_hotel' (Further cleanning is unreasonable for names)

In [16]:
gdf_working["hotel_type"] = (
    gdf_working["hotel_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

gdf_working["name"] = gdf_working["name"].fillna("unknown_hotel").astype("string")

gdf_working["name"].value_counts(dropna=False).to_frame("count")

,count
name,
unknown_hotel,11
Motel One,3
Living Hotel Weißensee,2
Hotel Amelie,2
Mit-Mensch,2
...,...
Lekkerurlaub,1
City Hotel am Kurfürstendamm,1
Hotel-Pension am Savignyplatz,1


### 🧮 Normalize `room_count`

The `room_count` field may contain inconsistent values depending on the source tag format.
To ensure the dataset is database-ready and consistent, we standardize `room_count` to a nullable integer:
- valid numeric values are kept
- non-numeric / descriptive strings are set to NULL

In [17]:
# Normalize room_count to a nullable integer (invalid/non-numeric -> NULL)
gdf_working["room_count"] = (
    pd.to_numeric(gdf_working["room_count"], errors="coerce")
    .round(0)
    .astype("Int64")
)

# Quick check
gdf_working["room_count"].value_counts(dropna=False)

room_count
<NA>    781
6         4
90        2
186       2
36        2
       ... 
69        1
50        1
48        1
19        1
45        1
Name: count, Length: 81, dtype: Int64

### Normalize star_rating

We normalize the `star_rating` column to a clean numeric format:

- Extract numeric values
- Accept only integers in the range **1–5**
- Any other value is set to **NA**

This step standardizes existing star ratings before any enrichment.
No external data is used here.

In [18]:
def normalize_star_rating(x):
    if pd.isna(x):
        return pd.NA
    try:
        v = int(float(x))
    except (ValueError, TypeError):
        return pd.NA
    return v if 1 <= v <= 5 else pd.NA

gdf_working["star_rating"] = gdf_working["star_rating"].apply(normalize_star_rating)

# Quick check
gdf_working["star_rating"].value_counts(dropna=False).sort_index()

star_rating
1         1
2        10
3        64
4        41
5         9
<NA>    756
Name: count, dtype: int64

### Consolidate contact fields (website / phone / email)

OSM may store contact information in multiple columns (e.g. `contact:website` vs `website`).
We consolidate to a single field per contact type using precedence:

1) `contact:*`
2) fallback field (e.g. `website`, `phone`, `email`)

In [19]:
# Consolidate contact fields (contact:* takes precedence)

def coalesce(a, b):
    return a.fillna(b)

# Website
gdf_working["website"] = coalesce(
    gdf_working["contact_website"],
    gdf_working["website_raw"]
)

# Phone
gdf_working["phone"] = coalesce(
    gdf_working["contact_phone"],
    gdf_working["phone_raw"]
)

# Email
gdf_working["email"] = coalesce(
    gdf_working["contact_email"],
    gdf_working["email_raw"]
)

# Clean common string-null artifacts
for c in ["website", "phone", "email"]:
    gdf_working[c] = gdf_working[c].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

# ---- Example previews (3 rows each) ----
display(gdf_working[["contact_website", "website_raw", "website"]].head(10))
display(gdf_working[["contact_phone", "phone_raw", "phone"]].head(10))
display(gdf_working[["contact_email", "email_raw", "email"]].head(10))


contact_website  \
element id                                                            
node    29997721                                                NaN   
        58467591                                                NaN   
        60105926  https://www.living-hotels.com/hotel-grosser-ku...   
        60105927  http://www.artotels.com/berlin-hotel-de-d-1017...   
        60105928   https://www.living-hotels.com/hotel-berlin-mitte   
        60240143                                                NaN   
        60240155  http://www.novotel.com/de/hotel-3278-novotel-b...   
        66917101                                                NaN   
        66917107                                                NaN   
        66917193  https://www.nh-collection.com/hotel/nh-collect...   

                                                        website_raw  \
element id                                                            
node    29997721                                                NaN   
        58467591                                                NaN   
        60105926                                                NaN   
        60105927                                                NaN   
        60105928                                                NaN   
        60240143  https://www.radissonhotels.com/en-us/hotels/ra...   
        60240155                                                NaN   
        66917101                            https://neuerfritz.com/   
        66917107  https://www.melia.com/de/hotels/deutschland/be...   
        66917193                                                NaN   

                                                            website  
element id                                                           
node    29997721                                                NaN  
        58467591                                                NaN  
        60105926  https://www.living-hotels.com/hotel-grosser-ku...  
        60105927  http://www.artotels.com/berlin-hotel-de-d-1017...  
        60105928   https://www.living-hotels.com/hotel-berlin-mitte  
        60240143  https://www.radissonhotels.com/en-us/hotels/ra...  
        60240155  http://www.novotel.com/de/hotel-3278-novotel-b...  
        66917101                            https://neuerfritz.com/  
        66917107  https://www.melia.com/de/hotels/deutschland/be...  
        66917193  https://www.nh-collection.com/hotel/nh-collect...

contact_phone phone_raw            phone
element id                                                  
node    29997721              NaN       NaN              NaN
        58467591              NaN       NaN              NaN
        60105926    +49 30 246000       NaN    +49 30 246000
        60105927              NaN       NaN              NaN
        60105928  +49 30 24600900       NaN  +49 30 24600900
        60240143              NaN       NaN              NaN
        60240155      +4930206740       NaN      +4930206740
        66917101      +4930284900       NaN      +4930284900
        66917107  +49 30 20607900       NaN  +49 30 20607900
        66917193              NaN       NaN              NaN

contact_email               email_raw  \
element id                                                                     
node    29997721                                 NaN                     NaN   
        58467591                                 NaN                     NaN   
        60105926  grosserkurfuerst@living-hotels.com                     NaN   
        60105927                                 NaN                     NaN   
        60105928       berlinmitte@living-hotels.com                     NaN   
        60240143                                 NaN                     NaN   
        60240155                                 NaN                     NaN   
        66917101                                 NaN                     NaN   
        66917107                                 NaN  melia.berlin@melia.com   
        66917193                                 NaN                     NaN   

                                               email  
element id                                            
node    29997721                                 NaN  
        58467591                                 NaN  
        60105926  grosserkurfuerst@living-hotels.com  
        60105927                                 NaN  
        60105928       berlinmitte@living-hotels.com  
        60240143                                 NaN  
        60240155                                 NaN  
        66917101                                 NaN  
        66917107              melia.berlin@melia.com  
        66917193                                 NaN

### Latitude & longitude

All geometries are points. We extract latitude and longitude into explicit columns.

In [20]:
gdf_working["latitude"] = gdf_working.geometry.y
gdf_working["longitude"] = gdf_working.geometry.x

display(gdf_working[["geometry", "longitude", "latitude"]].head(3))

geometry  longitude   latitude
element id                                                       
node    29997721   POINT (13.27667 52.5015)  13.276670  52.501495
        58467591   POINT (13.4078 52.51146)  13.407803  52.511465
        60105926  POINT (13.40879 52.51225)  13.408788  52.512249

### Derive amenities & accessibility from OSM tags (with tag-value inspection)

Amenities and accessibility are derived from OSM tags in `gdf_osm_raw`
and assigned to `gdf_working` (index-aligned).

Before building the derived fields, we inspect the raw values found in the
relevant tag columns to define which values should be treated as "not present"

In [21]:
# Inspect raw values for the tag columns we will use
tag_cols_to_inspect = [
    # Wi-Fi related
    "internet_access",

    # Amenities booleans / presence
    "air_conditioning",
    "breakfast", "breakfast:buffet", "breakfast:diet:vegan",
    "restaurant", "bar", "spa", "sauna", "swimming_pool",
    "indoor_seating", "outdoor_seating",
    "dog",
    "smoking", "smoking:outside",

    # Accessibility
    "elevator",
    "wheelchair", "toilets:wheelchair", "rooms:wheelchair",
]

present_cols = [c for c in tag_cols_to_inspect if c in gdf_osm_raw.columns]

def show_top_values(col: str, n: int = 12) -> pd.DataFrame:
    s = gdf_osm_raw[col].astype("string").str.strip().str.lower()
    vc = s.value_counts(dropna=False).head(n)
    return vc.to_frame("count")

for c in present_cols:
    print(f"\n--- {c} (top values) ---")
    display(show_top_values(c, n=12))


--- internet_access (top values) ---


,count
internet_access,
<NA>,592
wlan,255
yes,32
no,2



--- air_conditioning (top values) ---


,count
air_conditioning,
<NA>,801
yes,70
no,10



--- breakfast (top values) ---


...

,count
wheelchair,
<NA>,317
yes,301
no,177
limited,86



--- toilets:wheelchair (top values) ---


,count
toilets:wheelchair,
<NA>,759
yes,106
no,16



--- rooms:wheelchair (top values) ---


,count
rooms:wheelchair,
<NA>,880
2,1


In [22]:
# ---------- helpers ----------
def _val(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.lower()

def _present_any(df: pd.DataFrame, cols: list[str], exclude: set[str] = {"no"}) -> pd.Series:
    present = pd.Series(False, index=df.index)
    for c in cols:
        if c in df.columns:
            v = _val(df[c])
            present = present | (v.notna() & (v != "") & (~v.isin(exclude)))
    return present.fillna(False)

def _present_yes(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return (_val(df[col]) == "yes").fillna(False)

def _present_in(df: pd.DataFrame, col: str, allowed: set[str]) -> pd.Series:
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return _val(df[col]).isin(allowed).fillna(False)

# Add labels into a per-row set (no duplicates by design)
def _add_set(ser_sets: pd.Series, mask: pd.Series, label: str) -> pd.Series:
    idx = ser_sets.index[mask]
    for i in idx:
        ser_sets.at[i].add(label)
    return ser_sets

# ---------- build amenities from gdf_osm_raw ----------
src = gdf_osm_raw  # IMPORTANT: use raw tags dataframe

amen_sets = pd.Series([set() for _ in range(len(src))], index=src.index, dtype="object")

amen_sets = _add_set(amen_sets, _present_any(src, ["internet_access"], exclude={"no"}), "Wi-Fi")
amen_sets = _add_set(amen_sets, _present_yes(src, "air_conditioning"), "Air conditioning")

breakfast_cols = ["breakfast", "breakfast:buffet", "breakfast:diet:vegan"]
amen_sets = _add_set(amen_sets, _present_any(src, breakfast_cols, exclude=set()), "Breakfast")

amen_sets = _add_set(amen_sets, _present_yes(src, "restaurant"), "Restaurant")
amen_sets = _add_set(amen_sets, _present_yes(src, "bar"), "Bar")
amen_sets = _add_set(amen_sets, _present_yes(src, "spa"), "Spa")
amen_sets = _add_set(amen_sets, _present_yes(src, "sauna"), "Sauna")
amen_sets = _add_set(amen_sets, _present_yes(src, "swimming_pool"), "Swimming pool")

amen_sets = _add_set(amen_sets, _present_yes(src, "indoor_seating"), "Indoor seating")
amen_sets = _add_set(amen_sets, _present_any(src, ["outdoor_seating"], exclude={"no"}), "Outdoor seating")

amen_sets = _add_set(amen_sets, _present_any(src, ["dog"], exclude={"no"}), "Pets allowed")

smoking_mask = (
    _present_in(src, "smoking", {"yes", "outside", "isolated", "separated"})
    | _present_yes(src, "smoking:outside")
)
amen_sets = _add_set(amen_sets, smoking_mask, "Smoking area")

# Convert sets -> sorted, comma-separated string
amen = amen_sets.apply(lambda s: ", ".join(sorted(s)) if len(s) else pd.NA)

# ---------- build accessibility from gdf_osm_raw ----------
acc_sets = pd.Series([set() for _ in range(len(src))], index=src.index, dtype="object")

acc_sets = _add_set(acc_sets, _present_yes(src, "elevator"), "Elevator")

wheelchair_mask = (
    _present_in(src, "wheelchair", {"yes", "limited"})
    | _present_yes(src, "toilets:wheelchair")
    | _present_any(src, ["rooms:wheelchair"], exclude=set())
)
acc_sets = _add_set(acc_sets, wheelchair_mask, "Wheelchair access")

acc = acc_sets.apply(lambda s: ", ".join(sorted(s)) if len(s) else pd.NA)

# ---------- assign into gdf_working (index-aligned) ----------
gdf_working["amenities"] = amen
gdf_working["accessibility"] = acc

In [23]:
gdf_working[["amenities", "accessibility"]].notna().sum()

amenities        304
accessibility    389
dtype: int64

In [24]:
# Preview hotels with derived amenities / accessibility (no NA)
gdf_working[
    ["name", "amenities", "accessibility"]
].dropna(subset=["amenities", "accessibility"]).head(5)

name                     amenities  \
element id                                                                 
node    60240155      Novotel Berlin Mitte              Air conditioning   
        66917101  Hotel Neuer Fritz Berlin                         Wi-Fi   
        69226092        Grand Hyatt Berlin  Air conditioning, Bar, Wi-Fi   
        84644762           Numa Berlin Arc                         Wi-Fi   
        84644772               Ibis Budget       Air conditioning, Wi-Fi   

                      accessibility  
element id                           
node    60240155  Wheelchair access  
        66917101  Wheelchair access  
        69226092  Wheelchair access  
        84644762  Wheelchair access  
        84644772  Wheelchair access

### Star rating enrichment from Wikidata (P10290)

OSM star ratings are often missing or inconsistently populated.
To improve coverage, we enrich missing `star_rating` values using Wikidata property **P10290**
(hotel rating → stars).

Rules:
- Only fill missing `star_rating` (do not overwrite existing values)
- Only accept numeric star values in the 1–5 range
- Enrichment is based on hotels that have a Wikidata QID (`wikidata_id`)
- The Wikidata results are stored as a CSV snapshot for reproducibility

In [25]:
# Candidates: rows with Wikidata id and missing star_rating
df_stars_candidates = gdf_working.loc[
    gdf_working["wikidata_id"].notna() & gdf_working["star_rating"].isna(),
    ["wikidata_id"]
].copy()

print("Total hotels:", len(gdf_working))
print("Hotels with Wikidata QID:", gdf_working["wikidata_id"].notna().sum())
print("Candidates needing stars:", len(df_stars_candidates))

df_stars_candidates.head(3)

Total hotels: 881
Hotels with Wikidata QID: 192
Candidates needing stars: 146


wikidata_id
element id                  
node    60105927  Q111390189
        60240143  Q111388148
        60240155  Q111387924

#### Load Wikidata stars snapshot (cached CSV)

To keep the notebook reproducible, Wikidata query results are stored as a CSV snapshot.
By default, we load the snapshot from:

`WIKIDATA_STARS_PATH`

In [26]:
df_wikidata_stars_raw = pd.read_csv(WIKIDATA_STARS_PATH)

print("Wikidata stars snapshot rows:", len(df_wikidata_stars_raw))
df_wikidata_stars_raw.head(5)

Wikidata stars snapshot rows: 104


,hotel,hotelLabel,ratingLabel,stars
0,http://www.wikidata.org/entity/Q1630863,Dorint Kurfürstendamm Berlin,5-star hotel rating,5
1,http://www.wikidata.org/entity/Q15907201,Kempinski Hotel Bristol Berlin,5-star hotel rating,5
2,http://www.wikidata.org/entity/Q18020955,Hotel am Zoo,5-star hotel rating,5
3,http://www.wikidata.org/entity/Q46216294,Hotel Orania Berlin,5-star hotel rating,5
4,http://www.wikidata.org/entity/Q107187202,Steigenberger am Kanzleramt,5-star hotel rating,5


#### Clean Wikidata stars snapshot

The Wikidata snapshot contains:
- `hotel` as a full Wikidata entity URL (e.g. http://www.wikidata.org/entity/Q123)
- `stars` as the numeric star value

We extract the QID from `hotel`, keep only valid stars (1–5),
and prepare a clean lookup table for merging into `gdf_working`.

In [27]:
df_wikidata = df_wikidata_stars_raw[["hotel", "stars"]].copy()

# Extract QID from full URL
df_wikidata["wikidata_id"] = (
    df_wikidata["hotel"]
    .astype("string")
    .str.strip()
    .str.extract(r"(Q\d+)", expand=False)
)

# Normalize stars to int (1–5), else NA
df_wikidata["stars"] = pd.to_numeric(df_wikidata["stars"], errors="coerce").astype("Int64")
df_wikidata.loc[~df_wikidata["stars"].between(1, 5), "stars"] = pd.NA

df_wikidata_lookup = (
    df_wikidata[["wikidata_id", "stars"]]
    .dropna(subset=["wikidata_id", "stars"])
    .drop_duplicates(subset=["wikidata_id"], keep="first")
)

print("Lookup rows:", len(df_wikidata_lookup))
df_wikidata_lookup.head(5)

Lookup rows: 102


,wikidata_id,stars
0,Q1630863,5
1,Q15907201,5
2,Q18020955,5
3,Q46216294,5
4,Q107187202,5


#### Merge stars and update provenance (fill-only)

We merge stars by `wikidata_id` and fill only missing `star_rating`.
For rows that get filled, we update provenance fields:
- Append `;wikidata` to `data_source`
- Append `;wikidata:<QID>` to `source_ids`

In [28]:
before_missing = gdf_working["star_rating"].isna().sum()

gdf_working = gdf_working.merge(df_wikidata_lookup, on="wikidata_id", how="left")

# Identify rows that will be filled (missing before + Wikidata stars available)
fill_mask = gdf_working["star_rating"].isna() & gdf_working["stars"].notna()

# Fill only missing
gdf_working.loc[fill_mask, "star_rating"] = gdf_working.loc[fill_mask, "stars"].astype("Int64")

# Update provenance ONLY for filled rows
gdf_working.loc[fill_mask, "data_source"] = gdf_working.loc[fill_mask, "data_source"] + ";wikidata"
gdf_working.loc[fill_mask, "source_ids"] = (
    gdf_working.loc[fill_mask, "source_ids"] + ";wikidata:" + gdf_working.loc[fill_mask, "wikidata_id"].astype("string")
)

# Drop helper merge column
gdf_working = gdf_working.drop(columns=["stars"])

after_missing = gdf_working["star_rating"].isna().sum()

print("Missing star_rating before:", before_missing)
print("Missing star_rating after :", after_missing)
print("Filled by Wikidata        :", before_missing - after_missing)

# Quick preview of filled rows
gdf_working.loc[fill_mask, ["id", "name", "wikidata_id", "star_rating", "data_source", "source_ids"]].head(3)

Missing star_rating before: 756
Missing star_rating after : 654
Filled by Wikidata        : 102


,id,name,wikidata_id,star_rating,data_source,source_ids
3,60105927,art’otel Berlin Mitte,Q111390189,4,osm;wikidata,osm:60105927;wikidata:Q111390189
5,60240143,Radisson Collection Hotel Berlin,Q111388148,4,osm;wikidata,osm:60240143;wikidata:Q111388148
6,60240155,Novotel Berlin Mitte,Q111387924,4,osm;wikidata,osm:60240155;wikidata:Q111387924


### Fill missing address components via Nominatim (reverse geocoding)

We use Nominatim reverse geocoding to fill missing address components:
- `street`
- `house_number`
- `postal_code`

Rules:
- Fill only missing values (do not overwrite existing)
- Query using `latitude` and `longitude`
- Respect rate limits (sleep 1s per request)
- For rows that receive at least one filled component, append provenance:
  - `data_source += ";nominatim"`
  - `source_ids += ";nominatim:<place_id>"`

In [29]:
session = requests.Session()
session.headers.update({"User-Agent": NOMINATIM_USER_AGENT})

def nominatim_reverse(lat: float, lon: float) -> dict:
    params = {
        "lat": lat,
        "lon": lon,
        "format": "jsonv2",
        "addressdetails": 1,
        "zoom": 18,
    }
    r = session.get(NOMINATIM_URL, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def extract_addr_components(resp: dict):
    """
    Returns: (street, house_number, postal_code, place_id)
    Street fallback order in Nominatim address:
      road -> pedestrian -> footway -> path
    """
    addr = resp.get("address", {}) or {}

    street = (
        addr.get("road")
        or addr.get("pedestrian")
        or addr.get("footway")
        or addr.get("path")
    )
    house_number = addr.get("house_number")
    postal_code = addr.get("postcode")
    place_id = resp.get("place_id")

    street = street.strip() if isinstance(street, str) and street.strip() else pd.NA
    house_number = house_number.strip() if isinstance(house_number, str) and house_number.strip() else pd.NA
    postal_code = postal_code.strip() if isinstance(postal_code, str) and postal_code.strip() else pd.NA
    place_id = str(place_id).strip() if place_id is not None else pd.NA

    return street, house_number, postal_code, place_id

In [30]:
# Rows needing any of the 3 address parts
need_mask = gdf_working[["street", "house_number", "postal_code"]].isna().any(axis=1)
to_query = gdf_working.loc[need_mask, ["latitude", "longitude"]].copy()

print("Rows needing Nominatim:", len(to_query))

# Containers for results (aligned to index)
res_street = pd.Series(pd.NA, index=to_query.index, dtype="string")
res_house  = pd.Series(pd.NA, index=to_query.index, dtype="string")
res_post   = pd.Series(pd.NA, index=to_query.index, dtype="string")
res_place  = pd.Series(pd.NA, index=to_query.index, dtype="string")

# Live reverse geocoding
for i, (idx, row) in enumerate(to_query.iterrows(), start=1):
    lat = float(row["latitude"])
    lon = float(row["longitude"])

    try:
        resp = nominatim_reverse(lat, lon)
        street, house, post, place_id = extract_addr_components(resp)

        res_street.at[idx] = street
        res_house.at[idx]  = house
        res_post.at[idx]   = post
        res_place.at[idx]  = place_id

    except Exception as e:
        # Keep NA on failure; we do not stop the pipeline
        pass

    time.sleep(NOMINATIM_SLEEP_SEC)

print("Nominatim responses collected:", res_place.notna().sum())

Rows needing Nominatim: 188
Nominatim responses collected: 188


In [31]:
# Determine which fields will actually be filled (no overwrite)
fill_street = gdf_working["street"].isna() & res_street.notna()
fill_house  = gdf_working["house_number"].isna() & res_house.notna()
fill_post   = gdf_working["postal_code"].isna() & res_post.notna()

# Rows where at least one component was filled
filled_any = fill_street | fill_house | fill_post

# Fill only missing
gdf_working.loc[fill_street, "street"] = res_street.loc[fill_street]
gdf_working.loc[fill_house,  "house_number"] = res_house.loc[fill_house]
gdf_working.loc[fill_post,   "postal_code"] = res_post.loc[fill_post]

# Append provenance only for rows that received something
gdf_working.loc[filled_any, "data_source"] = gdf_working.loc[filled_any, "data_source"] + ";nominatim"
gdf_working.loc[filled_any, "source_ids"] = (
    gdf_working.loc[filled_any, "source_ids"] + ";nominatim:" + res_place.loc[filled_any].astype("string")
)

print("Filled street:", int(fill_street.sum()))
print("Filled house_number:", int(fill_house.sum()))
print("Filled postal_code:", int(fill_post.sum()))
print("Rows with any fill:", int(filled_any.sum()))

# Preview
gdf_working.loc[filled_any, ["name", "street", "house_number", "postal_code", "data_source", "source_ids"]].head(3)

Filled street: 158
Filled house_number: 45
Filled postal_code: 177
Rows with any fill: 181


,name,street,house_number,postal_code,data_source,source_ids
0,Rasthof AVUS,AVUS,NaN,14055,osm;nominatim,osm:29997721;nominatim:133080601
12,NH Collection Berlin Mitte am Checkpoint Charlie,Leipziger Straße,106-111,10117,osm;wikidata;nominatim,osm:69226077;wikidata:Q111388241;nominatim:133...
20,Hotel Gendarm,Charlottenstraße,63,10117,osm;nominatim,osm:86001831;nominatim:133146508


### Build address string

After filling missing address components, we construct a human-readable
`address` field using:

- `street`
- `house_number`
- `postal_code`

Rules:
- Use only available components
- Do not introduce placeholders
- Resulting format is consistent and readable

In [32]:
def build_address(row):
    parts = []

    if pd.notna(row["street"]):
        if pd.notna(row["house_number"]):
            parts.append(f'{row["street"]} {row["house_number"]}')
        else:
            parts.append(row["street"])

    if pd.notna(row["postal_code"]):
        parts.append(row["postal_code"])

    return ", ".join(parts) if parts else pd.NA


gdf_working["address"] = gdf_working.apply(build_address, axis=1)

# Quick sanity check
print("Missing street:", gdf_working["street"].isna().sum())
print("Missing house_number:", gdf_working["house_number"].isna().sum())
print("Missing postal_code:", gdf_working["postal_code"].isna().sum())
print("Missing address:", gdf_working["address"].isna().sum())

gdf_working[["name", "street", "house_number", "postal_code", "address"]].dropna(subset=["address"]).head(3)

Missing street: 0
Missing house_number: 120
Missing postal_code: 0
Missing address: 0


,name,street,house_number,postal_code,address
0,Rasthof AVUS,AVUS,NaN,14055,"AVUS, 14055"
1,Park Plaza Wallstreet,Wallstraße,23,10179,"Wallstraße 23, 10179"
2,Living Hotel Großer Kurfürst,Neue Roßstraße,11-12,10179,"Neue Roßstraße 11-12, 10179"


### LOR (Ortsteile) spatial enrichment

We enrich each hotel with Berlin administrative areas using the official
LOR Ortsteile geometry.

Method:
- Point-in-polygon spatial join
- Each hotel is assigned to exactly one Ortsteil

Derived fields:
- `district`        ← LOR `BEZIRK`
- `neighborhood`    ← LOR `OTEIL`
- `neighborhood_id` ← LOR `spatial_name`

Additionally:
- `district_id` is created using official Berlin district codes

In [33]:
# Load LOR Ortsteile polygons
gdf_lor = gpd.read_file(LOR_PATH)

# Ensure matching CRS (both should already be EPSG:4326)
gdf_lor = gdf_lor.to_crs(gdf_working.crs)

# Keep only required LOR columns
gdf_lor_sel = gdf_lor[["BEZIRK", "OTEIL", "spatial_name", "geometry"]].copy()

# Spatial join: point-in-polygon
gdf_working = gpd.sjoin(
    gdf_working,
    gdf_lor_sel,
    how="left",
    predicate="within"
)

# Rename columns to final schema
gdf_working = gdf_working.rename(
    columns={
        "BEZIRK": "district",
        "OTEIL": "neighborhood",
        "spatial_name": "neighborhood_id",
    }
)

# Drop spatial join helper column
gdf_working = gdf_working.drop(columns=["index_right"])

unmatched = gdf_working["district"].isna().sum()
total = len(gdf_working)

print("Total rows:", total)
print("Unmatched rows (no LOR polygon):", unmatched)
print("Match rate:", round((total - unmatched) / total * 100, 2), "%")

Total rows: 881
Unmatched rows (no LOR polygon): 0
Match rate: 100.0 %


In [34]:
# Create district_id using official codes
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

gdf_working["district_id"] = gdf_working["district"].map(district_mapping).astype("string")

# Quick sanity check
gdf_working[["name", "district", "district_id", "neighborhood", "neighborhood_id"]].dropna().head(3)

,name,district,district_id,neighborhood,neighborhood_id
0,Rasthof AVUS,Charlottenburg-Wilmersdorf,11004004,Westend,0405
1,Park Plaza Wallstreet,Mitte,11001001,Mitte,0101
2,Living Hotel Großer Kurfürst,Mitte,11001001,Mitte,0101


### Deduplication by proximity (10m) and normalized name

During enrichment and normalization, the same real-world hotel can appear more than once in the dataset (e.g., multiple OSM elements or closely located duplicate records).  
To ensure a single record per hotel, we apply a **proximity-based deduplication** rule:

**Dedup rule**
- Two records are considered duplicates if:
  - they share the same **normalized name** (lowercased + trimmed), and
  - their point geometries are within **10 meters** of each other.

In [35]:
# --- Prepare ---
before_rows = len(gdf_working)

# Normalize name for comparison
gdf_working["_name_norm"] = (
    gdf_working["name"]
    .astype("string")
    .str.lower()
    .str.strip()
)

# Project to metric CRS for distance checks (meters)
gdf_m = gdf_working.to_crs(epsg=3857)

# Container for indices to drop
to_drop = set()

# --- Proximity-based deduplication ---
for name, grp in gdf_m.groupby("_name_norm"):
    if len(grp) <= 1:
        continue

    coords = grp.geometry
    idxs = list(grp.index)

    # Compare pairwise distances
    for i in range(len(idxs)):
        if idxs[i] in to_drop:
            continue
        for j in range(i + 1, len(idxs)):
            if idxs[j] in to_drop:
                continue
            if coords.iloc[i].distance(coords.iloc[j]) <= 10:
                # Duplicate found → drop the later one
                to_drop.add(idxs[j])

print("Rows before:", before_rows)
print("Rows marked as duplicates (10m + same name):", len(to_drop))

# Show duplicate pairs
if to_drop:
    dup_indices = set(to_drop)

    # Add their nearest same-name neighbors
    for idx in to_drop:
        name_norm = gdf_m.loc[idx, "_name_norm"]
        grp = gdf_m[gdf_m["_name_norm"] == name_norm]
        dup_indices.update(grp.index.tolist())

    display(
        gdf_working.loc[
            gdf_working.index.isin(dup_indices),
            ["id", "name", "latitude", "longitude", "data_source", "source_ids"]
        ]
        .sort_values(["name", "latitude", "longitude"])
    )

# Drop duplicates
gdf_working = gdf_working.drop(index=list(to_drop)).copy()

after_rows = len(gdf_working)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)

# Cleanup helper column
gdf_working = gdf_working.drop(columns=["_name_norm"])

# Sanity check
print("Remaining rows with same name within 10m:",
      "OK" if len(to_drop) == (before_rows - after_rows) else "CHECK")

Rows before: 881
Rows marked as duplicates (10m + same name): 2


,id,name,latitude,longitude,data_source,source_ids
505,5486308620,Die Fabrik,52.498735,13.446034,osm,osm:5486308620
624,13075677601,die Fabrik,52.498712,13.446076,osm;nominatim,osm:13075677601;nominatim:415527444
606,12026653938,sly Berlin,52.524910,13.448064,osm,osm:12026653938
336,2844490311,sly Berlin,52.524920,13.448140,osm,osm:2844490311


Rows after: 879
Rows removed: 2
Remaining rows with same name within 10m: OK


### Final schema alignment

We align the transformed dataset to the final target schema by:
- selecting the required columns
- ordering them consistently

In [36]:
final_cols = [
    "id", "name", "hotel_type", "star_rating",
    "amenities", "accessibility", "room_count",
    "phone", "website", "email",
    "address", "street", "house_number", "postal_code",
    "latitude", "longitude", "geometry",
    "district", "neighborhood", "district_id", "neighborhood_id",
    "data_source", "source_ids"
]

# Keep only final columns, in order
gdf_final = gdf_working[final_cols].copy()

# Quick sanity check
print("Final shape:", gdf_final.shape)
gdf_final.head(3)

Final shape: (879, 23)


,id,name,hotel_type,star_rating,amenities,accessibility,room_count,phone,website,email,...,postal_code,latitude,longitude,geometry,district,neighborhood,district_id,neighborhood_id,data_source,source_ids
0,29997721,Rasthof AVUS,motel,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,...,14055,52.501495,13.276670,POINT (13.27667 52.5015),Charlottenburg-Wilmersdorf,Westend,11004004,0405,osm;nominatim,osm:29997721;nominatim:133080601
1,58467591,Park Plaza Wallstreet,hotel,4,<NA>,<NA>,<NA>,NaN,NaN,NaN,...,10179,52.511465,13.407803,POINT (13.4078 52.51146),Mitte,Mitte,11001001,0101,osm,osm:58467591
2,60105926,Living Hotel Großer Kurfürst,hotel,4,<NA>,Wheelchair access,<NA>,+49 30 246000,https://www.living-hotels.com/hotel-grosser-ku...,grosserkurfuerst@living-hotels.com,...,10179,52.512249,13.408788,POINT (13.40879 52.51225),Mitte,Mitte,11001001,0101,osm,osm:60105926


### Final QA overview

We perform a lightweight quality check to:
- inspect the final schema and column types
- understand overall data completeness

In [37]:
# Schema overview
gdf_final.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 879 entries, 0 to 880
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   id               879 non-null    string  
 1   name             879 non-null    string  
 2   hotel_type       879 non-null    string  
 3   star_rating      227 non-null    object  
 4   amenities        304 non-null    object  
 5   accessibility    389 non-null    object  
 6   room_count       100 non-null    Int64   
 7   phone            478 non-null    object  
 8   website          620 non-null    object  
 9   email            239 non-null    object  
 10  address          879 non-null    object  
 11  street           879 non-null    object  
 12  house_number     759 non-null    object  
 13  postal_code      879 non-null    object  
 14  latitude         879 non-null    float64 
 15  longitude        879 non-null    float64 
 16  geometry         879 non-null    geometry

In [38]:
# Percentage of missing values per column
missing_pct_per_col = (
    gdf_final.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .to_frame("missing_percentage")
)

missing_pct_per_col

,missing_percentage
room_count,88.62
star_rating,74.18
email,72.81
amenities,65.42
accessibility,55.75
phone,45.62
website,29.47
house_number,13.65
id,0.00
geometry,0.00


In [39]:
# Overall missingness percentage
total_cells = gdf_final.shape[0] * gdf_final.shape[1]
missing_cells = gdf_final.isna().sum().sum()
missing_pct = (missing_cells / total_cells) * 100

print(f"\nOverall missing values: {missing_cells} / {total_cells} "
      f"({missing_pct:.2f}%)")


Overall missing values: 3916 / 20217 (19.37%)


## Data transformation summary

This notebook builds a clean and enriched **Hotels – Berlin** dataset using **OpenStreetMap (OSM)** as the authoritative source, and supplements missing values with carefully scoped enrichments.

- **Extracted hotel-related POIs from OSM**
  - Eligible tags include `tourism=hotel`, `hostel`, `guest_house`, `aparthotel`.
  - Geometries were standardized to **Point** features in **EPSG:4326 (WGS84)** for consistent spatial processing and downstream database loading.

- **Normalized core fields**
  - Renamed raw OSM columns into the unified hotels schema (e.g. `hotel_type`, `star_rating`, `room_count`, contact fields, address parts).
  - Normalized `star_rating` to integers **1–5**; all invalid values were set to `NA`.

- **Created stable identifiers + provenance tracking**
  - Kept the OSM identifier as the stable `id`.
  - Initialized provenance fields:
    - `data_source` (e.g. `OSM`, `OSM;wikidata`, `OSM;nominatim`)
    - `source_ids` (e.g. `osm:<id>;wikidata:<QID>;nominatim:<place_id>`)

- **Derived amenities & accessibility (OSM-only)**
  - Built `amenities` and `accessibility` from raw OSM tags (no external enrichment available that is reliable and license-compatible).
  - Tag inspection showed inconsistent coverage across records, so we treat these fields as “best-effort” based on what is present in OSM.

- **Enriched missing star ratings via Wikidata (P10290)**
  - For hotels with a known `wikidata_id` and missing `star_rating`, we used Wikidata property **P10290 (hotel rating → stars)**.
  - The SPARQL results were exported as a static snapshot:
    - `hotels/sources/wikidata_stars_candidates.csv`
  - Fill policy: **only fill missing values; never overwrite existing OSM values.**

- **Filled missing address components via Nominatim reverse geocoding**
  - Used Nominatim (OSM) reverse geocoding to fill missing:
    - `street`, `house_number`, `postal_code`
  - Fill policy: **only fill missing values; do not overwrite existing values.**
  - Built `address` string **after** enrichment to ensure it reflects the best available components.

- **Administrative area enrichment via Berlin LOR**
  - Used official Berlin LOR Ortsteile polygons:
    - `mapping/lor_ortsteile.geojson`
  - Spatial join populated:
    - `district`, `neighborhood`, `neighborhood_id`
  - Created `district_id` using official Berlin district codes.

- **Deduplication (same name within 10 meters)**
  - Removed duplicates using:
    - normalized hotel name match + spatial proximity threshold (**10m**)  
  - Deduplication was performed at the end to ensure we keep the most complete enriched record.

### Output

The final dataset is stored in `gdf_final` and conforms to the unified POI requirements (id/name/lat/lon/geometry + district/neighborhood fields), with hotels-specific enrichment fields appended.

## Step 3: Load Hotels Data into DEV Database (`berlin_source_data.hotels`)

This step creates the final `berlin_source_data.hotels` table in the **DEV** database and loads the fully cleaned/enriched dataset from `gdf_final`.

### Prerequisites (must be done before running this step)
- AWS SSO session is active (if not: run `aws sso login --profile <your_profile>`).
- The secure tunnel is running via `./connect-db.sh` and shows **"Waiting for connections..."**.
- The tunnel exposes the DEV database locally at **`localhost:5433`**.

### What happens in this step
1. Create the table `berlin_source_data.hotels` (if it does not exist), including constraints and foreign keys.
2. Insert rows from `gdf_final` into the table.
3. Run quick validation checks (row count, duplicates, FK integrity).

⚠️ **One-time execution:** This step performs direct inserts into the DEV database. Do **not** re-run it once data is successfully loaded unless we explicitly decide on a reload strategy.

In [ ]:
import os
from sqlalchemy import create_engine, text

# DEV DB credentials 
user_name = ""
password = ""

# Connection (via SSM tunnel)
host = "localhost"
port = "5433"
database = ""
schema = ""

engine = create_engine(
    f"postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}"
)

print("Connection to DEV database established")

Connection to DEV database established


In [44]:
# 2) Create table (explicit schema — DEV, schema-aware)
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {schema}.hotels (
  id              VARCHAR(20)  PRIMARY KEY,
  district_id     VARCHAR(20)  NOT NULL,
  name            VARCHAR(200) NOT NULL DEFAULT 'unknown_hotel',
  latitude        DECIMAL(9,6),
  longitude       DECIMAL(9,6),
  geometry        VARCHAR,
  neighborhood    VARCHAR(100),
  district        VARCHAR(100),
  neighborhood_id VARCHAR(20),

  hotel_type      VARCHAR(50),
  star_rating     INT,
  amenities       TEXT,
  accessibility   VARCHAR(100),
  room_count      INT,
  phone           VARCHAR(100),
  website         VARCHAR(200),
  email           VARCHAR(200),
  address         VARCHAR(200),
  street          VARCHAR(200),
  house_number    VARCHAR(50),
  postal_code     VARCHAR(20),
  data_source     TEXT,
  source_ids      TEXT,

  CONSTRAINT hotels_district_fk
    FOREIGN KEY (district_id)
    REFERENCES {schema}.districts(district_id)
    ON DELETE RESTRICT
    ON UPDATE CASCADE,

  CONSTRAINT hotels_star_rating_check
    CHECK (star_rating IS NULL OR (star_rating BETWEEN 1 AND 5))
);
"""

with engine.begin() as conn:
    conn.execute(text(create_table_sql))

print(f"Table {schema}.hotels is ready")

Table berlin_source_data.hotels is ready


In [46]:
# 3) Insert data
# Insert final cleaned dataset into DEV DB (schema-aware)

df_insert = gdf_final.copy()

# Ensure geometry is inserted as text (WKT), not shapely objects
df_insert["geometry"] = df_insert["geometry"].apply(
    lambda g: g.wkt if g is not None else None
)

print(f"Rows prepared for insert: {len(df_insert)}")

df_insert.to_sql(
    name="hotels",
    con=engine,
    schema=schema,          
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Data insert completed successfully")

/var/folders/y5/h_wdy2hn5l153q25x6jslg3r0000gn/T/ipykernel_23482/2388356596.py:7: UserWarning: Geometry column does not contain geometry.
  df_insert["geometry"] = df_insert["geometry"].apply(


Rows prepared for insert: 879
Data insert completed successfully


In [48]:
# 4) Validation checks (schema-aware)
with engine.connect() as conn:
    row_count = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.hotels;")).scalar()

    dup_ids = conn.execute(text(f"""
        SELECT COUNT(*) FROM (
            SELECT id FROM {schema}.hotels GROUP BY id HAVING COUNT(*) > 1
        ) t;
    """)).scalar()

    invalid_district_fk = conn.execute(text(f"""
        SELECT COUNT(*)
        FROM {schema}.hotels h
        LEFT JOIN {schema}.districts d
          ON h.district_id = d.district_id
        WHERE d.district_id IS NULL;
    """)).scalar()

print("📊 Validation results:")
print(f"   • Rows in table: {row_count}")
print(f"   • Duplicate ID groups: {dup_ids}")
print(f"   • Invalid district_id FK rows: {invalid_district_fk}")
print("✅ Validation completed")

📊 Validation results:
   • Rows in table: 879
   • Duplicate ID groups: 0
   • Invalid district_id FK rows: 0
✅ Validation completed
